# Data Preprocessing



1. Handling Missing Values
2. Removing Duplicates
3. Data Conversion and Formatting
4. Data Integration
5. Scaling
6. Discretization





## Handling missing values

In [ ]:
import pandas as pd
import numpy as np

# Create dummy data
data = {
    'Name': ["Alice", "Bob", "", "Diane"],
    'Age': [25, np.nan, 30, 28],
    'Salary': [50000, None, 60000, 52000]
}
df = pd.DataFrame(data)

print("Original DataFrame:")
df.head()

In [ ]:
df.info()

The `.info()` method is a very easy and quick way to check for missing entries. But it won't work for string-entries of any type, so that it here misses the empty entry in the "Name" column! For Pandas, this column has 4 entries and all are "non-null" which means "not missing". Simply because there *is* a value in every row, even if one of them is a probably undesired `""`.

### Identifying All Types of Missing Values

Pandas' `isnull()` and `isna()` methods effectively identify `NaN` and `None` values. However, they do not recognize empty strings (`''`) as missing. We need to explicitly check for these separately, especially in string-type columns.

In [ ]:
# Identify NaN and None values across the DataFrame
missing_nan_none = df.isnull().sum()
print("Count of NaN/None values per column:")
print(missing_nan_none)

In [ ]:
missing_nan_none_per_row = df.isnull().sum(axis=1)
print("Count of NaN/None values per row:")
print(missing_nan_none_per_row)

In [ ]:
# Identify empty strings in the 'Name' column
missing_empty_strings = df[df['Name'] == '']
print("Rows with empty strings in 'Name' column:")
display(missing_empty_strings)

In [ ]:
# Combine all identified missing values (NaN, None, and empty strings)
# Create a boolean mask for rows with any NaN/None value
any_nan_none_row = df.isnull().any(axis=1)

# Create a boolean mask for rows with empty strings in 'Name'
any_empty_string_name_row = (df['Name'] == '')                  #| (df['Name'] == "missing")

# Combine both masks to find all rows with any type of missing data
all_missing_data_rows = df[any_nan_none_row | any_empty_string_name_row]

print("All rows identified with any type of missing data:")
display(all_missing_data_rows)

### Method 1: Remove Missing Values

In [ ]:
# Removing any rows that have at least one missing value
clean_df = df.dropna()
print("DataFrame after dropna():")
clean_df.head()

We could also use a simple mask to remove names that we found to be represented by certain strings.

In [ ]:
[x not in ["", "unknown", "missing", "None"] for x in df.Name]

In [ ]:
mask = [x not in ["", "unknown", "missing", "None"] for x in df.Name]
df[mask]

In [ ]:
# Removing any rows that have at least one missing value
clean_df = df[mask]
clean_df = clean_df.dropna()
print("DataFrame after deletions:")
clean_df.head()

### Method 2: Imputation

Imputation means that we alter, or rather fake data. So, obviously, we should be extremely careful in how and when we do this in practice.

In [ ]:
# Create randomly generated raw data
raw_data = []
for i in range(100):
    name = f"Person_{i+1}"
    age = np.random.randint(18, 66)  # Age between 18 and 65
    salary = np.random.randint(30000, 100000) # Salary between 30000 and 100000
    raw_data.append([name, age, salary])

data = pd.DataFrame(raw_data, columns=["Name", "Age", "Salary"])
data

Let us now remove some data to see what we could do to later fix this.

In [ ]:
data.iloc[1:4, 2] = np.nan
data.head()

Often people will simply fill missing entries with a global value such as the mean or median.

In [ ]:
# Find median of the salary
median_salary = np.median(data.dropna().Salary)
median_salary

In [ ]:
data_imputed = data.fillna(median_salary)
data_imputed.head()

If this is now "only" 3 entries out of 100 and we are not primarily interested in the Salary for further analysis it could be OK to do something like this. Still, always remember that we just invented new values, so this data should not be kept or shared without making this clear.

Technically often a bit better than simple mean or median substitutions are imputations based on existing data, for instance using the kNN-imputer from Scikit-Learn. We will later, in the machine-learning part see in depth how such a model works. Here we will skip the details and just say that it will replace missing values by what is found in the *k* most similar existing entries.

In [ ]:
from sklearn.impute import KNNImputer

# Creating the imputer object
imputer = KNNImputer(n_neighbors=2)

# Copy all data
data_imputed = data.copy()

# Replace the columns used for imputation (the imputor object will only change missing values and leave existing ones unchanged!)
data_imputed[['Age', 'Salary']] = imputer.fit_transform(data[['Age', 'Salary']])

print("DataFrame after KNN Imputation:")
data_imputed.head()

If you now go back to our original data you will see that those guesses are still off, but better than just taking the global median value.

## Removing Duplicates

Duplicate entries in a dataset can skew analysis and lead to incorrect conclusions. It’s often important to identify and remove duplicates to ensure data integrity.

In [ ]:
# Sample data with duplicate entries
data = pd.DataFrame({
    'CustomerID': [1, 2, 2, 3, 4, 4, 4],
    'Name': ['Alice', 'Bob', 'Bob', 'Charlie', 'Dave', 'Dave', 'Dave'],
    'PurchaseAmount': [250, 150, 150, 300, 400, 400, 400]
})

print("Original Data:")
data.head()

In [ ]:
# Removing duplicates
data_unique = data.drop_duplicates()
print("Data after Removing Duplicates:")
data_unique.head()

## Data Conversion and Formatting

Data types and formatting play a critical role in the accuracy and efficiency of data analysis. Incorrect data types or inconsistent formatting can lead to errors or misinterpretations during data processing. For instance, numeric values stored as strings may not be usable for calculations without conversion, and date strings may be misinterpreted if their format is not uniformly recognized. Ensuring data is correctly typed and formatted is crucial for:

- Data Quality: Correct types and formats ensure that the data adheres to the expected standards, improving the overall quality and reliability of the dataset.

- Analytical Accuracy: Proper data types are essential for performing accurate mathematical and statistical calculations.

- Operational Efficiency: Operations on data, such as sorting and indexing, are more efficient when data types are appropriately set.

In [ ]:
# Sample data with incorrect data types
data = pd.DataFrame({
    'ProductID': ['001', '002', '003'],
    'Price': ['12.5', '15.0', '20.25'],
    'Date': ['2021-01-01', '2021-01-02', '2021-01-03']
})

# Convert 'ProductID' to integer
data['ProductID'] = data['ProductID'].astype(int)

# Convert 'Price' to float
data['Price'] = data['Price'].replace(',', '.').astype(float)

print("Data after conversion:")
print(data)

In [ ]:
# Convert 'Date' to datetime format
data['Date'] = pd.to_datetime(data['Date'], format='%Y-%m-%d')
print("Data after formatting Date:")
data.head()

In [ ]:
# Sample data with decimal confusion
sales_data = pd.DataFrame({
    'Region': ['US', 'EU', 'IN'],
    'Sales': ['10,000.50', '8.000,75', '1,000.00']
})

# Correcting decimal delimiters based on region
sales_data['Sales'] = sales_data.apply(
    lambda row: row['Sales'].replace('.', '').replace(',', '.') if row['Region'] == 'EU' else row['Sales'].replace(',', ''),
    axis=1
).astype(float)

print("Corrected Sales Data:")
sales_data.head()

In [ ]:
# Sample data with string inconsistencies
customer_data = pd.DataFrame({
    'CustomerName': [' Alice ', 'bob', 'CHARLES']
})

# Standardizing string format: trim, lower case, capitalize
customer_data['CustomerName'] = customer_data['CustomerName'].str.strip().str.lower().str.capitalize()

print("Standardized Customer Names:")
customer_data.head()

## Data Integration

In many real-world data science scenarios, data doesn't come in a single, comprehensive package. Instead, relevant information is often scattered across multiple datasets. Combining these datasets is a crucial step as it enables a holistic analysis, providing a more complete view of the data subjects. Whether it's merging customer information from different branches of a company, integrating sales data from various regions, or linking patient data from multiple clinical studies, effectively combining datasets can yield insights that aren't observable in isolated data.

At first, this seems to a be a rather simple operations. In practice, however, this is often surprisingly complicated and critical. If merging is not done correctly, we might either lose data or create incorrect entries.

The basic syntax of the `merge` function is:

```python
pd.merge(left_data, right_data, how='inner', on=None, left_on=None, right_on=None)
```

With the following key parameters:
- `how`: The type of merge to be performed. The options include:

    - 'inner': Returns rows with matching keys in both DataFrames.
    - 'left': Returns all rows from the left DataFrame, along with matching rows from the right DataFrame.
    - 'right': Returns all rows from the right DataFrame, along with matching rows from the left DataFrame.
    - 'outer': Returns all rows from both DataFrames. Missing values will be filled with NaN.

- `on`: Column or list of columns to join on. The column(s) must be present in both DataFrames.

- `left_on` and `right_on`: These are used when the keys to merge on have different names in the left and right DataFrames.

In [ ]:
# Create sample data for products
data_products = pd.DataFrame({
    'ProductID': [101, 102, 103, 104],
    'ProductName': ['Widget', 'Gadget', 'DoesWhat', 'DoSomething']
})

# Create sample data for sales
data_sales = pd.DataFrame({
    'ProductID': [101, 102, 104, 105],
    'UnitsSold': [134, 243, 76, 100]
})
data_products.head()

In [ ]:
data_sales.head()

In [ ]:
inner_join_df = pd.merge(data_products, data_sales, on='ProductID', how='inner')
print("Inner Join Result:")
inner_join_df.head()

In [ ]:
left_outer_join_df = pd.merge(data_products, data_sales, on='ProductID', how='left')
print("Left Outer Join Result:")
left_outer_join_df.head()

In [ ]:
full_outer_join_df = pd.merge(data_products, data_sales, on='ProductID', how='outer')
print("Full Outer Join Result:")
full_outer_join_df.head()

The given example is fairly simple. In practice, you will often face far more complex situation that often require specific work-arounds. Some very common challenges for merging data are:

- **Duplicates**: Data can have repeated or conflicting entries.
- **Inconsistent Nomenclature**: Consistency in naming conventions can save hours of data wrangling later on. A classic example being different formats of names. "First Name Last Name" versus "Last Name, First Name".

It is therefore often a good strategy to work with a good identifier system where every datapoint is linked to one specific, unique ID. This can be a number, a hash, a specific code, a filename etc.

## Scaling

Scaling is a technique used in data preprocessing to standardize the range of independent variables or features of data. This is particularly important for many machine learning algorithms that are sensitive to the magnitude of the input features, such as K-Nearest Neighbors, Support Vector Machines, and neural networks.

There are two common types of scaling:

1.  **MinMax Scaling (Normalization)**: This method scales data to a fixed range, usually 0 to 1. It transforms features by scaling each feature to a given range. The formula for MinMax scaling is:
    `X_scaled = (X - X_min) / (X_max - X_min)`

2.  **Standard Scaling (Standardization)**: This method transforms data to have a mean of 0 and a standard deviation of 1. It is useful when the data follows a Gaussian distribution or when the algorithm assumes zero mean and unit variance. The formula for Standard scaling is:
    `X_scaled = (X - μ) / σ`, where `μ` is the mean and `σ` is the standard deviation.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Create sample data
data = {
    'Feature1': [10, 20, 30, 40, 50],
    'Feature2': [100, 200, 300, 400, 500],
    'Feature3': [1, 2, 3, 4, 5]
}
df_scaling = pd.DataFrame(data)

print("Original DataFrame:")
display(df_scaling)

### MinMax Scaling (Normalization)

In [ ]:
# Initialize MinMaxScaler
scaler_minmax = MinMaxScaler()

# Apply MinMax scaling to the DataFrame
df_minmax_scaled = pd.DataFrame(scaler_minmax.fit_transform(df_scaling), columns=df_scaling.columns)

print("DataFrame after MinMax Scaling:")
display(df_minmax_scaled)

### Standard Scaling (Standardization)

In [ ]:
# Initialize StandardScaler
scaler_standard = StandardScaler()

# Apply Standard scaling to the DataFrame
df_standard_scaled = pd.DataFrame(scaler_standard.fit_transform(df_scaling), columns=df_scaling.columns)

print("DataFrame after Standard Scaling:")
display(df_standard_scaled)

## Discretization

Discretization, also known as binning, is the process of transforming continuous numerical variables into discrete categorical bins. This technique is often used in data preprocessing for several reasons:

*   **Simplification**: It can help to simplify the data by reducing the number of unique values for a given feature, which can be useful for algorithms that prefer categorical data.
*   **Handling Outliers**: Binning can mitigate the impact of outliers by placing them into the same bin as other data points, rather than letting them disproportionately affect the model.
*   **Improving Model Performance**: For certain models, like decision trees, discretizing features can sometimes improve performance by making decision boundaries clearer.

Here we look at two discretization techniques:

1.  **Equal-Width Binning (Fixed-Width Binning)**: Divides the range of the data into `k` bins of equal width. The width of each bin is determined by `(max_value - min_value) / k`.
2.  **Equal-Frequency Binning (Quantile Binning)**: Divides the data into `k` bins, where each bin contains approximately the same number of data points. This is achieved by using quantiles.

In [ ]:
import pandas as pd
import numpy as np

# Create sample data for discretization
data = {
    'Income': np.random.normal(loc=50000, scale=15000, size=1000)
}
df_discretization = pd.DataFrame(data)

print("Original Income Data (first 5 rows):")
display(df_discretization.head())

### Equal-Width Binning

In [ ]:
# Apply equal-width binning to 'Income' into 5 bins
df_discretization['Income_EqualWidth_Bins'] = pd.cut(df_discretization['Income'], bins=5)

print("DataFrame after Equal-Width Binning (first 5 rows):")
display(df_discretization.head())

print("Value counts for Equal-Width Bins:")
print(df_discretization['Income_EqualWidth_Bins'].value_counts().sort_index())

### Equal-Frequency Binning

In [ ]:
# Apply equal-frequency binning to 'Income' into 5 bins
df_discretization['Income_EqualFreq_Bins'] = pd.qcut(df_discretization['Income'], q=5)

print("DataFrame after Equal-Frequency Binning (first 5 rows):")
display(df_discretization.head())

print("Value counts for Equal-Frequency Bins:")
print(df_discretization['Income_EqualFreq_Bins'].value_counts().sort_index())